#### Libraries, utils

In [ ]:
import numpy as np
import joblib
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import pandas as pd

# -----------------------
# Activation functions
# -----------------------
def phi(x):
    """tanh activation function."""
    return np.tanh(x)

def dphi(x):
    """Derivative of tanh activation."""
    return 1.0 - np.tanh(x)**2

# -----------------------
# Interaction (coupling) matrix
# -----------------------
def inter_matrix(K, N, g, gamma):
    """
    Fast interaction matrix with correlation gamma between reciprocal elements.
    """
    z0 = np.random.normal(size=(N, N))
    z1 = np.random.normal(size=(N, N))

    J = np.zeros((N, N))
    scale = g / np.sqrt(K)

    iu = np.triu_indices(N, k=1)
    J[iu] = scale * z0[iu]
    J[(iu[1], iu[0])] = scale * (gamma * z0[iu] + np.sqrt(1 - gamma**2) * z1[iu])

    return J

# -----------------------
# Adjacency matrix
# -----------------------
def adjacency_matrix(N, K, degree_sequence):
    """
    Generates a symmetric adjacency matrix with given degree sequence.
    """
    p = np.outer(degree_sequence, degree_sequence) / (K * N)
    p = np.clip(p, 0, 1)

    A = (np.random.rand(N, N) < p).astype(float)
    A = np.triu(A, 1)
    A = A + A.T
    np.fill_diagonal(A, 0)

    return A

# -----------------------
# Log-normal sampling
# -----------------------
def sample_lognormal(mu, sigma, size=1):
    """
    Samples from a log-normal distribution. If sigma is near zero, returns constant values.
    """
    if sigma < 1e-3:
        return np.ones(size) * np.exp(mu)
    else:
        return np.random.lognormal(mean=mu, sigma=sigma, size=size)

# -----------------------
# Coupling & eigenvalue analysis
# -----------------------
def coupling_matrix(N, g, gamma, mu, sigma, n):
    """
    Computes the adjacency-weighted interaction matrix W and returns the largest
    real eigenvalue (shifted by identity) along with parameters.
    """
    degree_sequence = sample_lognormal(mu, sigma, size=N)
    K = int(np.mean(degree_sequence))

    A = adjacency_matrix(N, K, degree_sequence)
    J = inter_matrix(K, N, g, gamma)
    W = A * J

    # Shift by identity for stability analysis
    eigenvalues = np.linalg.eigvals(W - np.eye(N))
    l_max = np.real(eigenvalues).max()

    return [g, gamma, sigma, n, l_max]

#### Maximum real part of eigenvalues analysis

In [ ]:
# -------------------------------
# Parameters
# -------------------------------
gamma_list = np.linspace(0, 1, 11)
sigma_list = [1]
N, mu = 4000, 3
N_samples = 100

# Estimate total number of simulations
N_params = len(gamma_list) * len(sigma_list) * 9 * N_samples
print(f'Total number of simulations: {N_params}')

# -------------------------------
# Prepare parameter combinations
# -------------------------------
params = []
for gamma in gamma_list:
    for sigma in sigma_list:
        g_c_theory = 1 / ((1 + gamma) * np.exp(sigma**2 / 2))
        for delta_g in np.linspace(-0.2, 0.2, 9):
            g = g_c_theory + delta_g
            for n in range(N_samples):
                params.append((N, g, gamma, mu, sigma, n))

# -------------------------------
# Run simulations in parallel
# -------------------------------
results = joblib.Parallel(n_jobs=-1, verbose=11)(
    joblib.delayed(coupling_matrix)(N, g, gamma, mu, sigma, n)
    for (N, g, gamma, mu, sigma, n) in params
)

# -------------------------------
# Save results
# -------------------------------
df = pd.DataFrame(results, columns=['g', 'gamma', 'sigma', 'replica', 'l_max'])
df.to_csv(f'largest_eigenvalue_N{N}_mu{mu}.csv', index=False)

# -------------------------------
# Load results and aggregate
# -------------------------------
df = pd.read_csv(f'largest_eigenvalue_N{N}_mu{mu}.csv')

df_grouped = df.groupby(['gamma', 'sigma', 'g']).agg(
    l_max_mean=('l_max', 'mean'),
    l_max_std=('l_max', 'std')
).reset_index()

# -------------------------------
# Zero-crossing interpolation
# -------------------------------
def interpolate_zero_crossing_with_error(group):
    """
    Interpolates the zero crossing of l_max_mean vs g.
    Returns first zero crossing and an error estimate.
    """
    group = group.sort_values('g')
    g_vals = group['g'].values
    y_vals = group['l_max_mean'].values

    zeros, errors = [], []

    for i in range(len(y_vals) - 1):
        y1, y2 = y_vals[i], y_vals[i + 1]

        if y1 == 0:
            zeros.append(g_vals[i])
            errors.append(0.0)
        elif y1 * y2 < 0:
            g1, g2 = g_vals[i], g_vals[i + 1]
            g0 = g1 + (0 - y1) * (g2 - g1) / (y2 - y1)
            zeros.append(g0)
            errors.append(abs(g2 - g1) / 2)  # half interval as error

    if zeros:
        # return first zero crossing
        return pd.Series({'g_zero': zeros[0], 'g_zero_error': errors[0]})
    else:
        return pd.Series({'g_zero': np.nan, 'g_zero_error': np.nan})

# Apply zero-crossing to each group
result = df_grouped.groupby(['gamma', 'sigma']).apply(interpolate_zero_crossing_with_error).reset_index()

# Add theoretical g (APPROXIMATE VALUE)
result['g_theo'] = 1 / ((1 + result['gamma']) * np.exp(result['sigma']**2 / 2))

g_zero_df = result